# Mean-centered SVD of household food consumption

The decomposition operates on a household x food-item matrix. It is run on the
de-identified per-household records in `data/mira_cleaned_git.csv`; the
commune-level aggregate in `data/commune_food_monthly.csv` cannot substitute
for it, because aggregating first and then decomposing does not yield the same
components.

Commune-monthly PC1 scores derived here are released in
`data/commune_monthly.csv`, which `dc1_commune_models.ipynb` and
`svd_ndvi_commune_mean_centering.ipynb` use.

Communes are numbered 1-6 as in the manuscript; see `data/commune_lookup.csv`.
All paths are relative to this notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

from scipy import stats
from scipy import optimize
import seaborn as sns; sns.set() # set default plot styles
import emcee
import statsmodels.api as sm
import corner
import time
from scipy.optimize import curve_fit
from datetime import datetime, timedelta


In [ ]:
#Load data

#  Reaad in MIRA data
# De-identified household-level food consumption records, released with this
# repository (see ../data/README.md).
file = '../data/mira_cleaned_git.csv'
mira = pd.read_csv(file, delimiter=',') 
print(mira.columns)

# Read in commune EVI series (communes numbered 1-6 as in the manuscript;
# see ../data/commune_lookup.csv for names and original shapefile codes)
c1e=pd.read_csv('../data/evi_timeseries/commune_1_evi.csv')
c2e=pd.read_csv('../data/evi_timeseries/commune_2_evi.csv')
c3e=pd.read_csv('../data/evi_timeseries/commune_3_evi.csv')
c4e=pd.read_csv('../data/evi_timeseries/commune_4_evi.csv')
c5e=pd.read_csv('../data/evi_timeseries/commune_5_evi.csv')
c6e=pd.read_csv('../data/evi_timeseries/commune_6_evi.csv')

In [ ]:
# Assemble the commune EVI panel
print(c1e)

# Dates are released as ISO calendar dates -- no MATLAB serial-date conversion needed.
for df in [c1e, c2e, c3e, c4e, c5e, c6e]:
    df["date"] = pd.to_datetime(df["date"])

commune_evis = pd.DataFrame({
    "date": c1e["date"],
    "c1": c1e["evi"],
    "c2": c2e["evi"],
    "c3": c3e["evi"],
    "c4": c4e["evi"],
    "c5": c5e["evi"],
    "c6": c6e["evi"],
})


commune_evis["date"] = pd.to_datetime(commune_evis["date"])

commune_evis_monthly = (
    commune_evis
    .set_index("date")
    .sort_index()
    .resample("MS")
    .mean()
    .reset_index() 
)

print(commune_evis_monthly.head())


commune_evis_filtered = commune_evis_monthly[
    (commune_evis_monthly["date"] >= "2018-07-01") &
    (commune_evis_monthly["date"] <= "2022-03-31")
].copy()

commune_evis_filtered["month"] = commune_evis_filtered["date"].dt.month

print(commune_evis_filtered.head())




In [ ]:
# Prep data for SVD
#Replace nan by column averages for the appropriate commune 
#Pull food columns
food_columns = ['Maize', 'Sorghum', 'Millet', 'Rice', 'Cassava Root', 'Cassava Leaves', 'Sweet Potato',
                'Other Tubers', 'Peanuts', 'Cowpeas', 'Beans', 'Dolique', 'Other Legumues', 'green/white cactus',
                'red cactus', 'vegetables', 'melons', 'other fruit', 'cactus leaves', 'other leaves',
                'watermelon', 'peas']


mira[food_columns] = (
    mira.groupby('commune')[food_columns]
      .transform(lambda x: x.fillna(x.mean()))
)

# Create a df with just foods
ndvi_foods = mira[food_columns]

# Mean centering
ndvi_foods_centered = (
    mira
    .groupby('commune')[food_columns]
    .transform(lambda x: x - x.mean())
)


# Perform SVD on the mean-centered data
U, Sigma, VT = np.linalg.svd(ndvi_foods_centered, full_matrices=False)


# Plot the first principal component
plt.figure(figsize=[14,4])
plt.bar(food_columns, VT[0, :])  # First principal component
plt.title('First Principal Component (Mean-Centered Data)')
plt.xticks(rotation=90)
plt.ylabel('Loading')
plt.grid(True)
plt.show()

# Plot the second principal component
plt.figure(figsize=[14,4])
plt.bar(food_columns, VT[1, :])  # Second principal component
plt.title('Second Principal Component (Mean-Centered Data)')
plt.xticks(rotation=90)
plt.ylabel('Loading')
plt.grid(True)
plt.show()

# Plot the third principal component
plt.figure(figsize=[14,4])
plt.bar(food_columns, VT[2, :])  # Third principal component
plt.title('Third Principal Component (Mean-Centered Data)')
plt.xticks(rotation=90)
plt.ylabel('Loading')
plt.grid(True)
plt.show()

# monthly_averages.to_csv('monthly_averages.csv', index=False)  # index=False to not include the DataFrame index in the file


In [ ]:
# Project 1st PCA back onto the data
pc1 = VT[0, :]

projected_data = np.dot(ndvi_foods_centered, pc1)

# Add back to MIRA data
mira['dc1'] = projected_data

# print(mira['dc1'])
# Print the Variance Explained
total_variance = np.sum(Sigma**2)  # Sum of squares of all singular values
variance_explained_first_PCA = (Sigma[0]**2) / total_variance  # Proportion for the first singular value

print("Variance explained by the first principal component: {:.2f}%".format(variance_explained_first_PCA * 100))




In [ ]:
#Plot by communes 
# Add year month var to df
# avg ndvi by year month 
mira['YearMonth'] = pd.to_datetime(mira[['year', 'month']].assign(day=1))

In [ ]:
# Commune 1 - Marolinta

c1_mira = mira[mira['commune']==1]
avg_dc1_c1 = c1_mira.groupby('YearMonth')['dc1'].mean()

print(c1_mira.head())
# print(avg_dc1_c1.head())

# Left ax
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(avg_dc1_c1)
ax1.set_ylabel('Mean DC1')
ax1.set_xlabel('YearMonth')
ax1.tick_params(axis='x', rotation=45)

print(c1e)

# Right ax
ax2 = ax1.twinx()
ax2.plot(commune_evis_filtered['date'], commune_evis_filtered['c1'], color='green')
ax2.set_ylabel('Mean EVI')

plt.title('Commune 1 (Marolinta) DC1 vs EVI (Monthly Means)')
plt.tight_layout()
plt.show()



In [ ]:
# # COMMUNE 1 (Marolinta) Models
# print(avg_dc1_c1.head())
# print(commune_evis_filtered.head())


# # Setting index for join
df1_indexed = avg_dc1_c1.to_frame(name="dc1")                    
df2_indexed = commune_evis_filtered.set_index("date")

# Merge datasets for ols
c1_df = df1_indexed.join(df2_indexed, how="inner")

# print(c1_df.head())

import statsmodels.formula.api as smf

#simple ols
model = smf.ols('dc1 ~ c1', data=c1_df).fit()

# Print results
print(model.summary())

##### Month fixed effect

c1_mdf = smf.ols("dc1 ~ c1 + C(month)", data=c1_df).fit()

print('C1 Month Effect Model')
print(c1_mdf.summary())

In [ ]:
# Commune 2 - Tranovaho

c2_mira = mira[mira['commune']==2]
avg_dc1_c2 = c2_mira.groupby('YearMonth')['dc1'].mean()


# print(avg_dc1)

# Left ax
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(avg_dc1_c2)
ax1.set_ylabel('Mean DC1')
ax1.set_xlabel('YearMonth')
ax1.tick_params(axis='x', rotation=45)

# Right ax
ax2 = ax1.twinx()
ax2.plot(commune_evis_filtered['date'], commune_evis_filtered['c2'], color='green')
ax2.set_ylabel('Mean EVI')

plt.title('Commune 2 (Tranovaho) DC1 vs EVI (Monthly Means)')
plt.tight_layout()
plt.show()

In [ ]:
# COMMUNE 2 (Tranovaho) Models
print(avg_dc1_c2.head())
print(commune_evis_filtered.head())


# # Setting index for join
df1_indexed = avg_dc1_c2.to_frame(name="dc1")                    
df2_indexed = commune_evis_filtered.set_index("date")

# Merge datasets for ols
c2_df = df1_indexed.join(df2_indexed, how="inner")

print(c2_df.head())


#simple ols
model = smf.ols('dc1 ~ c2', data=c2_df).fit()

# Print results
print(model.summary())

##### Month fixed effect

c2_mdf = smf.ols("dc1 ~ c2 + C(month)", data=c2_df).fit()

print('C2 Month Effect Model')
print(c2_mdf.summary())

In [ ]:
# Commune 3 - Marovato
c3_mira = mira[mira['commune']==3]
avg_dc1_c3 = c3_mira.groupby('YearMonth')['dc1'].mean()


# print(avg_dc1)

# Left ax
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(avg_dc1_c3)
ax1.set_ylabel('Mean DC1')
ax1.set_xlabel('YearMonth')
ax1.tick_params(axis='x', rotation=45)

# Right ax
ax2 = ax1.twinx()
ax2.plot(commune_evis_filtered['date'], commune_evis_filtered['c3'], color='green')
ax2.set_ylabel('Mean EVI')

plt.title('Commune  DC1 vs EVI (Monthly Means)')
plt.tight_layout()
plt.show()

In [ ]:
# COMMUNE 3 (Marovato) Models
print(avg_dc1_c3.head())
print(commune_evis_filtered.head())


# # Setting index for join
df1_indexed = avg_dc1_c3.to_frame(name="dc1")                    
df2_indexed = commune_evis_filtered.set_index("date")

# Merge datasets for ols
c3_df = df1_indexed.join(df2_indexed, how="inner")

print(c3_df.head())


#simple ols
model = smf.ols('dc1 ~ c3', data=c3_df).fit()

# Print results
print(model.summary())

##### Month fixed effect

c3_mdf = smf.ols("dc1 ~ c3 + C(month)", data=c3_df).fit()

print('C3 Month Effect Model')
print(c3_mdf.summary())

In [ ]:
# Commune 4 - Anjampaly

c4_mira = mira[mira['commune']==4]
avg_dc1_c4 = c4_mira.groupby('YearMonth')['dc1'].mean()


# print(avg_dc1)

# Left ax
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(avg_dc1_c4)
ax1.set_ylabel('Mean DC1')
ax1.set_xlabel('YearMonth')
ax1.tick_params(axis='x', rotation=45)

print(c1e)

# Right ax
ax2 = ax1.twinx()
ax2.plot(commune_evis_filtered['date'], commune_evis_filtered['c4'], color='green')
ax2.set_ylabel('Mean EVI')

plt.title('Commune 4 (Anjampaly) DC1 vs EVI (Monthly Means)')
plt.tight_layout()
plt.show()

In [ ]:
# COMMUNE 4 (Anjampaly) Models
print(avg_dc1_c4.head())
print(commune_evis_filtered.head())


# # Setting index for join
df1_indexed = avg_dc1_c4.to_frame(name="dc1")                    
df2_indexed = commune_evis_filtered.set_index("date")

# Merge datasets for ols
c4_df = df1_indexed.join(df2_indexed, how="inner")

print(c4_df.head())

#simple ols
model = smf.ols('dc1 ~ c4', data=c4_df).fit()

# Print results
print(model.summary())

##### Month fixed effect

c4_mdf = smf.ols("dc1 ~ c4 + C(month)", data=c4_df).fit()

print('C4 Month Effect Model')
print(c4_mdf.summary())

In [ ]:
# Commune 5 - Antaritarika

c5_mira = mira[mira['commune']==5]
avg_dc1_c5 = c5_mira.groupby('YearMonth')['dc1'].mean()

# Left ax
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(avg_dc1_c5)
ax1.set_ylabel('Mean DC1')
ax1.set_xlabel('YearMonth')
ax1.tick_params(axis='x', rotation=45)

# Right ax
ax2 = ax1.twinx()
ax2.plot(commune_evis_filtered['date'], commune_evis_filtered['c5'], color='green')
ax2.set_ylabel('Mean EVI')

plt.title('Commune 5 (Antaritarika) DC1 vs EVI (Monthly Means)')
plt.tight_layout()
plt.show()

In [ ]:
# COMMUNE 5 Models - Antaritarika
print(avg_dc1_c5.head())
print(commune_evis_filtered.head())

# Setting index for join
df1_indexed = avg_dc1_c5.to_frame(name="dc1")
df2_indexed = commune_evis_filtered.set_index("date")

# Merge datasets for ols
c5_df = df1_indexed.join(df2_indexed, how="inner")

print(c5_df.head())

# simple ols
model = smf.ols('dc1 ~ c5', data=c5_df).fit()
print(model.summary())

##### Month fixed effect
c5_mdf = smf.ols("dc1 ~ c5 + C(month)", data=c5_df).fit()

print('C5 Month Effect Model')
print(c5_mdf.summary())

In [ ]:
# Commune 6 - Imongy

c6_mira = mira[mira['commune']==6]
avg_dc1_c6 = c6_mira.groupby('YearMonth')['dc1'].mean()

# Left ax
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(avg_dc1_c6)
ax1.set_ylabel('Mean DC1')
ax1.set_xlabel('YearMonth')
ax1.tick_params(axis='x', rotation=45)

# Right ax
ax2 = ax1.twinx()
ax2.plot(commune_evis_filtered['date'], commune_evis_filtered['c6'], color='green')
ax2.set_ylabel('Mean EVI')

plt.title('Commune 6 (Imongy) DC1 vs EVI (Monthly Means)')
plt.tight_layout()
plt.show()

In [ ]:
# COMMUNE 6 Models - Imongy
print(avg_dc1_c6.head())
print(commune_evis_filtered.head())

# Setting index for join
df1_indexed = avg_dc1_c6.to_frame(name="dc1")
df2_indexed = commune_evis_filtered.set_index("date")

# Merge datasets for ols
c6_df = df1_indexed.join(df2_indexed, how="inner")

print(c6_df.head())

# simple ols
model = smf.ols('dc1 ~ c6', data=c6_df).fit()
print(model.summary())

##### Month fixed effect
c6_mdf = smf.ols("dc1 ~ c6 + C(month)", data=c6_df).fit()

print('C6 Month Effect Model')
print(c6_mdf.summary())

In [ ]:
# Summary of commune-level slopes (EVI -> DC1), communes 1-6

rows = []
for c, df, mdf in [
    ('c1', c1_df, c1_mdf),
    ('c2', c2_df, c2_mdf),
    ('c3', c3_df, c3_mdf),
    ('c4', c4_df, c4_mdf),
    ('c5', c5_df, c5_mdf),
    ('c6', c6_df, c6_mdf),
]:
    simple = smf.ols(f'dc1 ~ {c}', data=df).fit()
    rows.append({
        'commune': c,
        'n': int(simple.nobs),
        'beta_simple': simple.params[c],
        'p_simple': simple.pvalues[c],
        'r2_simple': simple.rsquared,
        'beta_month_fe': mdf.params[c],
        'p_month_fe': mdf.pvalues[c],
        'r2_month_fe': mdf.rsquared,
    })

results_tbl = pd.DataFrame(rows).set_index('commune')
print(results_tbl.round(4))